# Find organisms in SeaTube

Query real ONC annotations, inspect the recorded taxa, and extract frames or short clips with their provenance. This walkthrough uses one ROV dive from 6 July 2019; change the dates and organism below for your own study.

Install the repository with `python -m pip install -e ".[notebooks]"`, then open this notebook in JupyterLab. Metadata queries require internet access and an ONC token. Frame and clip extraction also requires [ffmpeg](https://ffmpeg.org/download.html) on your PATH.

## 1. Get your ONC token

1. [Create an Oceans 3.0 account](https://data.oceannetworks.ca/Registration), then [sign in](https://data.oceannetworks.ca).
2. Open your [Profile](https://data.oceannetworks.ca/Profile) (top right in Oceans 3.0).
3. Select **Web Services API** and choose **Copy Token**. If no token exists, generate one first. See [ONC's token instructions](https://oceannetworkscanada.github.io/Oceans3.0-API/Home.html#how-to-obtain-an-onc-token).
4. Create a file named `.env` in the repository root (beside `pyproject.toml`) and add:

```dotenv
ONC_TOKEN=your-token-here
```

The repository ignores `.env`. Keep the token out of notebook cells, outputs, and commits. You can also set `ONC_TOKEN` in the environment; an existing environment value takes precedence over `.env`. The next cell loads the token without displaying it.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from seatube import AnnotationSet, SeaTube, ReviewFilters

# Works when Jupyter starts in the repository root or its examples directory.
ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file() and (p / "seatube").is_dir()),
    None,
)
if ROOT is None:
    raise RuntimeError("Start Jupyter from the seatube-downloader repository.")

load_dotenv(ROOT / ".env", override=False)
if not os.environ.get("ONC_TOKEN"):
    raise RuntimeError(
        "ONC_TOKEN is missing. Follow the token instructions above and add it to the repository's .env file."
    )

OUTPUT_DIR = ROOT / "downloads" / "research_walkthrough"
sea = SeaTube(token=os.environ["ONC_TOKEN"], data_dir=OUTPUT_DIR)
print("ONC token loaded; the next query will check access.")

## 2. Choose a search

The catalog describes supported searches. Counts later in the notebook show which groups were actually annotated in the selected dive.

`crabs` includes Brachyura and Anomura; `true-crabs` includes only Brachyura. You can also search by scientific name, such as `Chionoecetes tanneri` or `Sebastes`. See the [complete organism catalog](../docs/organisms.md) for aliases and group boundaries.

In [ ]:
START_DATE = "2019-07-06T00:00:00Z"
END_DATE = "2019-07-06T23:59:59Z"
ORGANISM = "crabs"
DOWNLOAD_MEDIA = False  # Set True after checking the plan; downloads one source archive.

for group in sea.groups("crab"):
    print(group["group"], "→", ", ".join(group["ancestors"]))

## 3. Discover a dive and fetch its annotations

List the dives in the date range, then select one by its returned ID. The default chooses the first dive, so this is an exploratory sample rather than a survey of the whole archive. The fetch below always contacts ONC and saves the returned metadata locally; it does not download video.

In [ ]:
dives = sea.dives(START_DATE, END_DATE)
if not dives:
    raise RuntimeError("No dives overlap these dates. Choose a different date range.")

for dive in dives[:10]:
    print(dive["diveId"], dive.get("referenceDiveId"), dive.get("dateFrom"))

selected_dive = dives[0]  # Change this after inspecting the list.
print("Selected dive:", selected_dive["diveId"], selected_dive.get("referenceDiveId"))

In [ ]:
annotations = sea.fetch(
    START_DATE, END_DATE,
    dive_ids={int(selected_dive["diveId"])},
    resolution="L",
    save_to=OUTPUT_DIR / "annotations.json",
)
print(annotations.summary())

# In later sessions, reuse the downloaded metadata:
# annotations = AnnotationSet.load(OUTPUT_DIR / "annotations.json")

## 4. Inspect the available organisms

The first classification pass may take a few minutes while WoRMS lineages are cached. Counts represent annotation records, not animals; groups overlap. `mapped_annotations` counts observations with a usable position inside a source video file. A taxonomy warning means some memberships could not be determined; inspect `sea.resolver.unresolved`.

In [ ]:
available = sea.available_groups(annotations)
for row in available:
    print(f"{row['group']:20} {row['annotations']:5} annotated, {row['mapped_annotations']:5} mapped")

print("Unresolved taxonomy entries:", len(sea.resolver.unresolved))

In [ ]:
for taxon in annotations.taxon_summary()[:15]:
    print(taxon.name, taxon.aphia_id, taxon.annotations)

## 5. Filter observations

A list of organisms means **OR**. Additional review, depth, date, and location filters mean **AND**. A species query cannot recover species identity from an observation labelled only to family or phylum. Review fields provide quality signals, not proof of a correct identification.

In [ ]:
matches = sea.search(annotations, ORGANISM)
print("Search:", ORGANISM)
print(matches.summary())

reviewed = matches.filter(review=ReviewFilters(reviewed_only=True))
print("Reviewed matching annotations:", len(reviewed))

# Optional depth restriction; records without depth are excluded:
# depth_subset = matches.filter(min_depth_m=500, max_depth_m=2000)

if not matches:
    print("No matches: inspect the taxon list, taxonomy warnings, dates, and selected dive.")

### Who left these annotations?

List the people in the fetched dataset, then list just those associated with your organism search. Names and ONC user IDs let you choose one or several people without relying on name spellings.

**Author (`creator`)** is the person who created an annotation. **Last editor (`modifier`)** is the person who most recently changed it. SeaTube's returned records do not include a complete list of named expert reviewers; a last editor is not necessarily a reviewer. For “annotations left by this person,” filter authors. If you want records last edited by someone, use the modifier filter explicitly. See [ONC's field descriptions](https://wiki.oceannetworks.ca/pages/viewpage.action?pageId=140247051).

In [ ]:
print("Authors across the fetched dive:")
for person in annotations.people_summary():
    print(person["user_id"], person["name"], person["annotations"])

# These lists now describe only the organism search (crabs by default).
creators = matches.people_summary(role="creator")
modifiers = matches.people_summary(role="modifier")
print("\nAuthors of matching annotations:")
for person in creators:
    print(person["user_id"], person["name"], person["annotations"])
print("\nLast editors of matching annotations (not confirmed reviewers):")
for person in modifiers:
    print(person["user_id"], person["name"], person["annotations"])

print("\nFirst five matching annotations — ID | author | last editor:")
for annotation in matches[:5]:
    print(annotation.id, "|", annotation.creator_name, "|", annotation.modifier_name)

### Choose one or more people

The example selects the first author from the table (sorted by annotation count). Replace `CREATOR_IDS` with the IDs you want, or use `None` for any author. A list matches **any** selected person; an empty list selects nothing.

`MODIFIER_IDS=None` leaves the last-editor field unrestricted. To restrict last editors too, set it to one or more IDs from their table. Author and editor conditions combine with **AND**. The resulting `selected` subset drives every frame, clip, and export below. Each people selection gets a separate output folder, with source archives shared to avoid repeat downloads. To keep all organism matches, set both selections to `None`.

In [ ]:
creator_ids = [p["user_id"] for p in creators if p["user_id"] is not None]
modifier_ids = [p["user_id"] for p in modifiers if p["user_id"] is not None]

CREATOR_IDS = creator_ids[:1]  # One author; [:2] selects the first two, or enter IDs from the table.
MODIFIER_IDS = None           # E.g. modifier_ids[:1] to restrict to the first last editor.

selected = matches.filter(creator_ids=CREATOR_IDS, modifier_ids=MODIFIER_IDS)
print("Selected authors:", selected.people_summary("creator"))
print("Selected last editors:", selected.people_summary("modifier"))
print("After people filters:", selected.summary())

# A separate last-editor-only example, without restricting authors:
last_editor_subset = matches.filter(modifier_ids=modifier_ids[:1])
print("Annotations last edited by the first listed editor:", len(last_editor_subset))

# Name searches also work, but IDs avoid ambiguous names:
# named = matches.filter(creator=creators[0]["name"])

# Keep each people selection's indexes separate; source archives remain shared.
def selection_tag(ids):
    return "all" if ids is None else "-".join(str(uid) for uid in sorted(set(ids))) or "none"

SELECTION_DIR = OUTPUT_DIR / f"authors_{selection_tag(CREATOR_IDS)}_editors_{selection_tag(MODIFIER_IDS)}"


### Exercise: compare broad and narrow crab searches

Compare `crabs` with `true-crabs`. A difference means some recorded taxa fall under the broader definition, not that the archive contains additional unique animals. Results depend on the selected dive.

In [ ]:
broad = sea.search(annotations, "crabs")
narrow = sea.search(annotations, "true-crabs")
print("Crab annotations:", len(broad), "True-crab annotations:", len(narrow))

## 6. Plan frames and clips

Plan up to three frames from the people-filtered observations in one source archive. Clips use those same observations and archive, so extracting both requires at most one source download. The file remains cached for reuse.

ONC serves whole source files, even for one frame. The size report may contain unknown sizes; `max_videos` limits files, not bytes. The planner favors files with more requested frames and is not an ecological sampling design. Overlapping clip intervals merge and stop at archive boundaries.

In [ ]:
frames = selected.frames(max_images=3, max_videos=1)
clip_annotations = AnnotationSet(a.raw for frame in frames for a in frame.annotations)
clips = clip_annotations.clips(before_seconds=2, after_seconds=3, max_clips=3, max_videos=1)

archive_dir = OUTPUT_DIR / "archives"
images = sea.image_downloader(SELECTION_DIR / "images", video_dir=archive_dir, keep_videos=True)
video = sea.clip_downloader(SELECTION_DIR / "clips", video_dir=archive_dir, keep_videos=True)

print(images.describe_plan(frames))  # HEAD requests only; no video download
for clip in clips:
    print(clip)
if not frames:
    print("No mapped frames: choose another organism or dive and check annotations.summary().")

## 7. Extract and inspect media

Set `DOWNLOAD_MEDIA = True` in the configuration cell and rerun from there once the plan suits your needs. Install ffmpeg first: `brew install ffmpeg` on macOS, `sudo apt install ffmpeg` on Ubuntu, or use the [Windows builds](https://ffmpeg.org/download.html).

Frames are JPGs; clips are silent H.264 MP4s. The CSV/JSONL indexes retain labels, annotation IDs, archive offsets, people and location metadata, and SeaTube links. Labels refer to observation instants; an organism may not remain visible throughout a clip.

In [ ]:
image_rows, clip_rows = [], []
if DOWNLOAD_MEDIA:
    if not frames:
        raise RuntimeError("The current search has no mapped frames to download.")
    image_rows = images.download(frames)
    clip_rows = video.download(clips)
    print(f"Extracted/indexed {len(image_rows)} frames and {len(clip_rows)} clips.")
else:
    print("Metadata and plans are ready. Set DOWNLOAD_MEDIA=True to extract the selected media.")

In [ ]:
if image_rows:
    from IPython.display import Image, display
    display(Image(filename=str(SELECTION_DIR / "images" / image_rows[0]["image_file"])))
    print("Source moment:", image_rows[0]["seatube_link"])
# Open MP4 files in SELECTION_DIR / "clips" to inspect movement.

## 8. Export the selected observations

Keep the raw annotation JSON and `.worms_cache.json` with your analysis. Flat tables contain one row per annotation/taxon pair. Media JSONL indexes include the full source records; review personal metadata before sharing them.

Zero matches describe this query, not the absence of organisms in the ocean. Missing annotation, coarse labels, unresolved taxonomy, and recording gaps can all limit a dataset. For machine-learning splits, keep neighboring frames from the same dive/archive together to avoid leakage. Follow [ONC's citation guidance](https://www.oceannetworks.ca/data/how-to-cite-onc/) and credit the source teams and taxonomy.

In [ ]:
selected.save(SELECTION_DIR / "selected_annotations.json")
selected.write_flat_csv(SELECTION_DIR / "selected_annotations.csv")
selected.write_flat_jsonl(SELECTION_DIR / "selected_annotations.jsonl")
print("Exported annotations:", len(selected))
print("Flat annotation/taxon rows:", len(selected.flatten()))
sea.close()